# segmentor2.py

## training

In [1]:
import leopardgecko.segmentor2 as lgs2
import numpy as np

In [2]:
import logging
#logging.basicConfig(level=logging.INFO)
logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s — %(name)s — %(levelname)s — %(funcName)s:%(lineno)d — %(message)s",
        )
import tifffile

In [3]:
N=184

#N=160  #(16*10)


single volume N,N,N

In [4]:
data_labels_fn = ("./test_data/TS_0005_crop.tif", "./test_data/TS_0005_ribos_membr_crop.tif")

datafn0, labelfn0 =data_labels_fn

data = tifffile.imread(datafn0)
labels = tifffile.imread(labelfn0)

data_crop = data[:N,:N,:N]
label_crop = labels[:N,:N,:N]

traindatas=[data_crop]
trainlabels=[label_crop]

In [5]:
data_crop.shape

(184, 184, 184)

In [6]:
print(lgs2.nn1_train_epochs)
print(lgs2.nn2_train_epochs)

10
10


In [7]:
lgs2.nn1_train_epochs=2 # debug low number
lgs2.nn2_train_epochs=2
lgs2.nn1_batch_size = 4
nn1_lr=1e-7
nn1_max_lr=1e-3

In [8]:
#lgs2.quick_new_and_train_one_unet_model_per_axis(traindatas, trainlabels)

lgs2.quick_new_and_train_2unets_z_xy_models(traindatas, trainlabels)

2024-11-26 17:30:18,938 — root — INFO — quick_new_and_train_2unets_z_xy_models:2028 — quick_new_and_train_one_unet_model_per_axis
2024-11-26 17:30:18,939 — root — INFO — update_nn1_models_from_generators:172 — update_NN1_models_from_generators()
2024-11-26 17:30:18,940 — root — INFO — update_nn1_models_from_generators:177 — 2 NN1 models to be created
2024-11-26 17:30:18,940 — root — INFO — create_nn1_ptmodel_from_class_generator:122 — create_nn1_ptmodel_from_class_generator()
2024-11-26 17:30:19,497 — root — INFO — create_nn1_ptmodel_from_class_generator:122 — create_nn1_ptmodel_from_class_generator()
2024-11-26 17:30:19,921 — root — INFO — update_nn2_model_from_generator:1152 — update_NN2_model_from_generator()
2024-11-26 17:30:19,922 — root — INFO — create_nn2_ptmodel_from_class_generator:1119 — create_nn2_ptmodel_from_class_generator()
2024-11-26 17:30:19,923 — root — INFO — create_nn2_ptmodel_from_class_generator:1127 — hid_layers_num_list: [4, 4]
2024-11-26 17:30:19,924 — root — I

RuntimeError: Runtime error occurred when testing a slice of data with shape (184, 184, 184) along axis [0] on the model 0. Try different volume size such as multiples of 32.

Save model

In [ ]:
import datetime
DATE=str(datetime.date.today())
TIME=f"{datetime.datetime.now().hour:02d}{datetime.datetime.now().minute:02d}"
fname_stem=f"{DATE}_{TIME}"
model_fn = f"{fname_stem}_model.lgsegm2"
model_fn

In [ ]:
lgs2.save_lgsegm2_model(model_fn)

## see training progress

In [ ]:
lgs2.last_train_nn1_progress

In [ ]:
for k,v in lgs2.last_train_nn1_progress.items():
    print(k,v)

In [ ]:
for k,v in lgs2.last_train_nn1_progress.items():
    print(k,v['test_results'])
    print(len(v['test_results']))

In [ ]:
import matplotlib.pyplot as plt

nmodels = len(lgs2.last_train_nn1_progress.keys())

fig, axs= plt.subplots(2,2)
for k,v in lgs2.last_train_nn1_progress.items():

    avg_loss = [ t['avg_loss'] for t in v['test_results']]
    avg_metrics = [ t['avg_metric'] for t in v['test_results']]
    axs[0,k].plot( avg_loss)
    axs[0,k].set_title(f"nn1 model:{k}, avg_loss")
    axs[0,k].set_xlabel("epoch")
    axs[1,k].plot(avg_metrics)
    axs[1,k].set_title(f"nn1 model:{k}, avg_metrics")
    axs[1,k].set_xlabel("epoch")

In [ ]:
lgs2.last_train_nn2_progress

In [ ]:
import matplotlib.pyplot as plt

nmodels = len(lgs2.last_train_nn1_progress.keys())
fig, axs= plt.subplots(1,2)

v=lgs2.last_train_nn2_progress

avg_loss = [ t['avg_loss'] for t in v['test_results']]
avg_metrics = [ t['avg_metric'] for t in v['test_results']]
axs[0].plot( avg_loss)
axs[0].set_title(f"nn2 avg_loss")
axs[0].set_xlabel("epoch")
axs[1].plot(avg_metrics)
axs[1].set_title(f"nn2 avg_metrics")
axs[1].set_xlabel("epoch")

In [ ]:
assert False

# Load model and predict

Recommended restart kernel

## setup

In [ ]:
import numpy as np
import leopardgecko.segmentor2 as lgs2
import tifffile
import napari

import logging
#logging.basicConfig(level=logging.INFO)
logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s — %(name)s — %(levelname)s — %(funcName)s:%(lineno)d — %(message)s",
                    force=True
                    )

In [ ]:
import glob

lg2_files = glob.glob("*.lgsegm2")

lg2_file_last = sorted(lg2_files, reverse=True)[0]
lg2_file_last

In [ ]:
lgs2.load_lgsegm2_model(lg2_file_last)

In [ ]:
val_data = tifffile.imread(r"test_data\TS_0005_crop_val.tif")
val_labels_gnd = tifffile.imread(r"test_data\TS_0005_ribos_membr_crop_val.tif")

val_data_l = [val_data]

In [ ]:
#Normalise
datavols_norm_list0 = lgs2.normalise_volumes(val_data_l)

In [ ]:
# NV=napari.Viewer()
# NV.add_image(val_data)
# NV.add_image(datavols_norm_list0[0], name="normalized")
# NV.add_labels(val_labels_gnd, name="ground truth lbl")

OK

In [ ]:
slice = datavols_norm_list0[0][128,:,:] #in case it is needed, and example slice

In [ ]:
lgs2.torch_device_str_nn2 = "cuda:0"

Working through function by function in segmentor.py to find what is going wrong

## Check single slice prediction

In [ ]:
lgs2.NN1_models[0]

In [ ]:
lgs2.nn1_axes_to_models_indices

In [ ]:
import matplotlib.pyplot as plt
plt.imshow(slice)

In [ ]:
model = lgs2.NN1_models[0]
model.eval()

In [ ]:
import torch
import torch.nn
x = torch.unsqueeze( torch.unsqueeze( torch.from_numpy(slice), dim=0) , dim=0).float().to("cuda:0")
X=model(x)
SM_func = torch.nn.Softmax(dim=1)
pred_probs_slice = SM_func(X)

In [ ]:
pred_probs_slice.shape

In [ ]:
plt.imshow(pred_probs_slice.detach().cpu().numpy()[0,0,:,:])

OK

## Check nn1_predict_slices_along_axis

In [ ]:
probs0, lbls0 = lgs2.predict_nn1_slices_along_axis(datavols_norm_list0[0], axis=0)

In [ ]:
NV=napari.Viewer()
NV.add_image(val_data)
NV.add_image(probs0, name="pred probs")

OK

## Check NN1_predict

In [ ]:
import tempfile
from pathlib import Path
tempdir_pred= tempfile.TemporaryDirectory()
path_out_results = Path(tempdir_pred.name)
path_out_results

In [ ]:
res_pd = lgs2.predict_nn1(datavols_norm_list0, path_out_results)

Check results in temporary folder

In [ ]:
# cleanup
del(tempdir_pred)

## Check full `predict()`

In [ ]:
pred_res = lgs2.predict(val_data_l)

In [ ]:
import napari
NV=napari.Viewer()
NV.add_image(val_data)
NV.add_labels(val_labels_gnd, name="lbls gnd")
NV.add_labels(pred_res[0], name="pred lbls")

OK